## Import 

In [1]:
import math
import pickle 
import warnings
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interp
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from tableone import TableOne

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
race_path  = "Extraction/Data/csvExtract/"
path_data  = "./covid/Data/EHR/"

### Reading Data

In [3]:
df_ehr = pd.read_csv(path_data + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_ehr.head(2)

In [4]:
print(df_ehr.stay_id.nunique())
print(df_ehr.shape)

10561
(240003, 545)


### Drop Repeated Rows & Keep Notes

In [5]:
all_columns = list(df_ehr.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]
text_columns = ['cxr_image', 'cxr_lung', 'cxr_note', 'radiology_note', 'discharge_note', 
                'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_ehr.drop(remove_columns, axis=1, inplace=True)
df_ehr = df_ehr.drop_duplicates()

In [6]:
print(df_ehr.stay_id.nunique())
print(df_ehr.shape)

10561
(240003, 290)


### Fix Age

In [7]:
df_ehr.loc[df_ehr['age'] >= 95, 'age'] = 95
df_ehr = df_ehr[df_ehr.age > 16]

### Take first hours of ICU of patients with more than 24 hour LoS

In [8]:
max_rows = df_ehr.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [9]:
def take_observation_window(df, observation_window):
    df = df.groupby('stay_id').head(observation_window).reset_index(drop=True)
    return df

In [10]:
df_ehr = take_observation_window(df_ehr, observation_window)

### Remove Short Stay

In [11]:
split_label = 'hospital_expire_flag'
label = 'icu_expire_flag'

In [12]:
df_ehr_id = df_ehr[df_ehr.Bins >= 22].stay_id.unique()
df = df_ehr[df_ehr.stay_id.isin(df_ehr_id)].copy()

### Variables Selection

In [13]:
selected_columns = ['subject_id', 'stay_id', 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
                    'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
                    'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                    'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
                    'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                    'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'Total GCS', 'Richmond-RAS Scale',
                    'age', 'gender', 'race', 
                    'icuLos_h', 'icu_expire_flag', 'hospital_expire_flag']

In [14]:
for col in selected_columns:
    temp_col = col + '_ind'
    
    if temp_col in list(df.columns):
        df.loc[df[temp_col] == 0, col] = np.nan

In [15]:
df = df[selected_columns]

In [16]:
df.head(3)

In [17]:
df_ehr = df.drop(['subject_id', 'stay_id', 'age', 'gender', 'race', 'icuLos_h', 'hospital_expire_flag'], axis=1)
df_demog = df[['subject_id', 'stay_id', 'age', 'gender', 'race', 'icuLos_h', 'icu_expire_flag']]

### Create TableOne

In [18]:
columns = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
            'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
            'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
            'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
            'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
            'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
            'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
            'Total GCS', 'Richmond-RAS Scale',
            'icu_expire_flag']

In [19]:
categorical = ['Richmond-RAS Scale']

In [20]:
IQR = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
        'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
        'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
        'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
        'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
        'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
        'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
        'Total GCS',]

In [21]:
EHR_table = TableOne(df_ehr, groupby='icu_expire_flag', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [22]:
EHR_table

Grouped by icu_expire_flag                                                                       
                                                                              Missing              Overall                    0                    1 P-Value
n                                                                                                   218130               200449                17681        
Heart Rate, median [Q1,Q3]                                                      17278     82.0 [71.0,95.0]     81.0 [70.0,94.0]    90.0 [76.0,104.5]  <0.001
SpO2, median [Q1,Q3]                                                            19660     97.0 [95.0,99.0]     97.0 [95.0,99.0]     97.0 [94.0,99.0]  <0.001
Oxygen Saturation, median [Q1,Q3]                                              209645     95.0 [73.0,97.0]     95.0 [73.0,97.0]     88.0 [70.0,96.0]  <0.001
Respiratory Rate, median [Q1,Q3]                                                23531     18.3 [15.3,22.0]     18.0 [15.0,22.0]     22.0 [18.0,27.0]  <0.001
Temperature, median [Q1,Q3]                                                    147907     36.8 [36.6,37.2]     36.8 [36.6,37.2]     36.8 [36.4,37.2]  <0.001
Non Invasive Blood Pressure mean, median [Q1,Q3]                                93128     81.0 [72.0,92.0]     82.0 [73.0,93.0]     76.0 [68.0,86.0]  <0.001
Non Invasive Blood Pressure systolic, median [Q1,Q3]                            92964  116.0 [103.0,132.0]  117.0 [103.0,132.0]   109.0 [97.0,124.0]  <0.001
Non Invasive Blood Pressure diastolic, median [Q1,Q3]                           92978     67.0 [58.0,78.0]     68.0 [59.0,78.0]     63.0 [54.0,72.0]  <0.001
Glucose, median [Q1,Q3]                                                        160580  133.0 [111.0,167.0]  132.0 [111.0,165.0]  143.0 [111.2,194.0]  <0.001
Creatinine, median [Q1,Q3]                                                     198453        1.0 [0.7,1.6]        1.0 [0.7,1.5]        1.7 [1.1,2.8]  <0.001
Base Excess, median [Q1,Q3]                                                    194923      -2.0 [-5.0,0.0]      -2.0 [-5.0,0.0]    -5.0 [-10.0,-1.0]  <0.001
BUN, median [Q1,Q3]                                                            198514     19.0 [13.0,33.0]     18.0 [12.0,31.0]     31.0 [20.0,52.0]  <0.001
Anion Gap, median [Q1,Q3]                                                      198542     12.0 [10.0,15.0]      12.0 [9.0,14.0]     15.0 [12.0,19.0]  <0.001
Bicarbonate, median [Q1,Q3]                                                    198452     22.0 [19.0,24.0]     22.0 [20.0,24.0]     20.0 [16.0,23.0]  <0.001
Lactate, median [Q1,Q3]                                                        200138        2.1 [1.4,3.3]        2.0 [1.4,3.0]        3.5 [1.9,7.2]  <0.001
Hemoglobin, median [Q1,Q3]                                                     192262      10.1 [8.6,11.8]      10.2 [8.6,11.8]       9.3 [7.9,11.4]  <0.001
Hematocrit, median [Q1,Q3]                                                     192262     31.0 [26.4,35.9]     31.1 [26.6,35.9]     28.9 [24.5,35.2]  <0.001
pH, median [Q1,Q3]                                                             193306        7.4 [7.3,7.4]        7.4 [7.3,7.4]        7.3 [7.2,7.4]  <0.001
Bilirubin, Direct, median [Q1,Q3]                                              210356        0.3 [0.1,1.0]        0.3 [0.1,0.9]        0.5 [0.2,1.9]  <0.001
pO2, median [Q1,Q3]                                                            194837   114.0 [72.0,184.0]   119.0 [75.0,195.0]    90.0 [62.0,132.0]  <0.001
pCO2, median [Q1,Q3]                                                           194885     41.0 [36.0,47.0]     41.0 [36.0,46.0]     42.0 [35.0,51.0]   0.004
AST, median [Q1,Q3]                                                            210880    42.0 [22.0,104.0]     39.0 [21.0,94.0]    67.0 [33.0,167.0]  <0.001
ALT, median [Q1,Q3]                                                            210717     29.0 [15.0,71.0]     27.0 [15.0,65.0]    41.

### Static Information

In [33]:
df_demog = df_demog.groupby('stay_id').head(1).reset_index(drop=True)

In [34]:
df_demog.head()

### Read Race Dictionary

In [36]:
with open(race_path + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary = pickle.load(f)

In [37]:
general_ethnicity_mapping = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNKNOWN': 'Unknown',
    'UNABLE TO OBTAIN': 'Unknown',
    'PATIENT DECLINED TO ANSWER': 'Unknown',
    
    'OTHER': 'Other',
    'MIDDLE EASTERN': 'Other',
    'MULTIPLE RACE/ETHNICITY': 'Other',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other',
    
    'ASIAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - SOUTH EAST ASIAN': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'AMERICAN INDIAN/ALASKA NATIVE': 'Native American',
    
    'BLACK/AFRICAN': 'Black/African American',
    'BLACK/CAPE VERDEAN': 'Black/African American',
    'BLACK/AFRICAN AMERICAN': 'Black/African American',
    'BLACK/CARIBBEAN ISLAND': 'Black/African American',
    
    'SOUTH AMERICAN': 'Hispanic/Latino',
    'HISPANIC OR LATINO': 'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - COLUMBIAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN': 'Hispanic/Latino'}

In [38]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['race'] = df['race'].map(inv_ethnicity_dict)
    
    return df

In [39]:
def categorize_ethnicity(df, new_mapping):
    
    df['race'] = df['race'].map(new_mapping)
    
    return df

In [40]:
df_demog = replace_ethnicity_with_names(df_demog, race_dictionary)
df_demog = categorize_ethnicity(df_demog, general_ethnicity_mapping)

In [41]:
df_demog['Ethnicity'] = 'Non Hispanic'
df_demog.loc[df_demog.race == 'Hispanic/Latino', 'Ethnicity'] = 'Hispanic'

In [42]:
df_demog.head()

In [43]:
print(df_demog.subject_id.nunique())
print(df_demog[df_demog.icu_expire_flag == 0].subject_id.nunique())
print(df_demog[df_demog.icu_expire_flag == 1].subject_id.nunique())

7554
6958
737


In [44]:
columns = ['age', 'gender', 'race', 'Ethnicity', 'icuLos_h', 'icu_expire_flag']

categorical = ['gender', 'race', 'Ethnicity']

IQR = ['age', 'icuLos_h']

In [45]:
demog_table = TableOne(df_demog, groupby='icu_expire_flag', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [46]:
demog_table

Grouped by icu_expire_flag                                                                  
                                                                   Missing            Overall                  0                   1 P-Value
n                                                                                        9094               8357                 737        
age, median [Q1,Q3]                                                      0   66.0 [55.0,76.0]   66.0 [55.0,75.0]    68.0 [58.0,78.0]  <0.001
gender, n (%)            1                                               0        5429 (59.7)        4985 (59.7)          444 (60.2)   0.783
                         2                                                        3665 (40.3)        3372 (40.3)          293 (39.8)        
race, n (%)              Asian                                           0          611 (6.7)          570 (6.8)            41 (5.6)  <0.001
                         Black/African American                                     549 (6.0)          504 (6.0)            45 (6.1)        
                         Hispanic/Latino                                            508 (5.6)          484 (5.8)            24 (3.3)        
                         Native American                                             58 (0.6)           54 (0.6)             4 (0.5)        
                         Other                                                      339 (3.7)          318 (3.8)            21 (2.8)        
                         Unknown                                                  2274 (25.0)        1997 (23.9)          277 (37.6)        
                         White                                                    4755 (52.3)        4430 (53.0)          325 (44.1)        
Ethnicity, n (%)         Hispanic                                        0          508 (5.6)          484 (5.8)            24 (3.3)   0.005
                         Non Hispanic                                             8586 (94.4)        7873 (94.2)          713 (96.7)        
icuLos_h, median [Q1,Q3]                                                 0  65.8 [38.4,129.1]  62.8 [37.5,120.0]  129.7 [55.5,266.4]  <0.001
[1] Chi-squared tests for the following variables may be invalid due to the low number of observations: race.

### Save Tables

In [47]:
# EHR_table.to_csv('./Results/TableONe_EHR_ICU_Mortality.csv')
# demog_table.to_csv('./Results/TableONe_DEMOG_ICU_Mortality.csv')